# Mini-system example to inspect optimization constraints

FINE translates the components of an energy system into a linear optimization
problem, built with [Pyomo](https://www.pyomo.org/). This notebook shows that
problem for a very small system, so that you can see which variables and
constraints FINE creates and how to read them. This helps to understand
FINE's formulation and to debug your own models.

The system has two regions and three hourly time steps. It contains:

- a cheap electricity source in Region A (operating cost 1 €/MWh),
- an expensive electricity source in Region B (operating cost 10 €/MWh),
- an electricity demand in both regions,
- a power line between the two regions.

Expected result: Source A supplies both regions, and the line carries
electricity from A to B. Source B is not built.

The constraints of each component class are defined in the FINE source code,
for example in `fine/sourceSink.py` and `fine/transmission.py`.

In [1]:
import fine as fn
import pandas as pd
import pyomo.environ as pyo

## Build the model

All costs are in Euro (`costUnit`) and electricity is in MW
(`commodityUnitsDict`). Cost parameters therefore have the units €/MW
(investment) and €/MWh (operation). Each of the three time steps is one hour long.

In [2]:
# Define the mini-system input data
locations = {"Region_A", "Region_B"}
commodities = {"electricity"}

commodity_unit_dict = {
    "electricity": "MW",
}

number_of_time_steps = 3
hours_per_time_step = 1

In [3]:
# Create the FINE energy system model
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=number_of_time_steps,
    commodityUnitsDict=commodity_unit_dict,
    hoursPerTimeStep=hours_per_time_step,
    costUnit="Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

In [4]:
# Define the time series
time_steps = range(number_of_time_steps)

source_a_availability = pd.DataFrame(
    {
        "Region_A": [1.0, 1.0, 1.0],
        "Region_B": [0.0, 0.0, 0.0],
    },
    index=time_steps,
)

source_b_availability = pd.DataFrame(
    {
        "Region_A": [0.0, 0.0, 0.0],
        "Region_B": [1.0, 1.0, 1.0],
    },
    index=time_steps,
)

electricity_demand = pd.DataFrame(
    {
        "Region_A": [2.0, 3.0, 2.0],
        "Region_B": [4.0, 5.0, 6.0],
    },
    index=time_steps,
)

electricity_demand

,Region_A,Region_B
0,2.0,4.0
1,3.0,5.0
2,2.0,6.0


## Add the sources

Each source can only be built in its own region. `locationalEligibility`
alone would be enough for this; the zero values of `capacityMax` and of the
availability in the other region have the same effect.

- `investPerCapacity`: investment cost in €/MW,
- `opexPerOperation`: operating cost in €/MWh,
- `operationRateMax`: available share of the capacity in each time step (1 = full capacity).

In [5]:
# Add the electricity source in Region A
region_a_eligibility = pd.Series(
    {
        "Region_A": 1,
        "Region_B": 0,
    }
)

source_a_capacity_max = pd.Series(
    {
        "Region_A": 20.0,
        "Region_B": 0.0,
    }
)

esM.add(
    fn.Source(
        esM=esM,
        name="Source_A",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=region_a_eligibility,
        operationRateMax=source_a_availability,
        capacityMax=source_a_capacity_max,
        investPerCapacity=5,
        opexPerOperation=1,
        interestRate=0.08,
        economicLifetime=20,
    )
)

In [6]:
# Add the electricity source in Region B
region_b_eligibility = pd.Series(
    {
        "Region_A": 0,
        "Region_B": 1,
    }
)

source_b_capacity_max = pd.Series(
    {
        "Region_A": 0.0,
        "Region_B": 20.0,
    }
)

esM.add(
    fn.Source(
        esM=esM,
        name="Source_B",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=region_b_eligibility,
        operationRateMax=source_b_availability,
        capacityMax=source_b_capacity_max,
        investPerCapacity=5,
        opexPerOperation=10,
        interestRate=0.08,
        economicLifetime=20,
    )
)

In [7]:
# Add the electricity demand
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity_demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=electricity_demand,
    )
)

## Add the power line

A transmission component needs an eligibility matrix (which regions are
connected) and the distances between the regions, here 10 km.

`losses` is given **per km**. With 0.01 per km and 10 km, 10 % of the
transported electricity is lost, so only 90 % arrives.
`investPerCapacity` is also per km: 1 €/(MW·km).

In [8]:
# Define the transmission connection
transmission_eligibility = pd.DataFrame(
    [
        [0, 1],
        [1, 0],
    ],
    index=["Region_A", "Region_B"],
    columns=["Region_A", "Region_B"],
)

transmission_distances = pd.DataFrame(
    [
        [0, 10],
        [10, 0],
    ],
    index=["Region_A", "Region_B"],
    columns=["Region_A", "Region_B"],
)

transmission_eligibility

,Region_A,Region_B
Region_A,0,1
Region_B,1,0


In [9]:
# Add the transmission component
esM.add(
    fn.Transmission(
        esM=esM,
        name="Electricity_grid",
        commodity="electricity",
        hasCapacityVariable=True,
        locationalEligibility=transmission_eligibility,
        distances=transmission_distances,
        capacityMax=10,
        losses=0.01,
        investPerCapacity=1,
        opexPerOperation=0,
        interestRate=0.08,
        economicLifetime=40,
    )
)

In [10]:
# Show the components
esM.componentNames

{'Source_A': 'SourceSinkModel',
 'Source_B': 'SourceSinkModel',
 'Electricity_demand': 'SourceSinkModel',
 'Electricity_grid': 'TransmissionModel'}

## Optimize

`STANDARD_SOLVER` is Gurobi if FINE finds a full Gurobi licence, and the
free solver HiGHS otherwise. Time series aggregation is switched off, so all
three time steps are optimized.

In [11]:
# Optimize the system
esM.optimize(
    timeSeriesAggregation=False,
    solver=fn.utils.ImplementedSolvers.STANDARD_SOLVER.value,
)

Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")


CPU model: AMD Ryzen 9 7950X 16-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 16 physical cores, 32 logical processors, using up to 3 threads


... (15 outputs left out)

       0    0.0000000e+00   1.415000e+01   0.000000e+00      0s


       5    6.9116671e+04   0.000000e+00   0.000000e+00      0s


Solved in 5 iterations and 0.00 seconds (0.00 work units)


Optimal objective  6.911667094e+04


## Inspect the Pyomo model

After optimization, the Pyomo model is stored in `esM.pyM`.

FINE names variables as `<variable>_<component class>`:

| Prefix | Meaning |
|---|---|
| `cap` | installed capacity [MW] |
| `op` | operation in one time step [MWh] |
| `commis` / `decommis` | capacity built / removed in an investment period [MW] |
| `nbReal` / `nbInt` | number of installed units (real / integer) |
| `commisBin` | binary build decision |

The suffix `_srcSnk` stands for sources and sinks, `_trans` for transmission.

The index of an operation variable, for example `('Region_A', 'Source_A', 0, 0, 2)`,
is (location, component, investment period, period, time step).
For transmission, the location is a pair: `Region_A_Region_B` is the flow from A to B.

In [12]:
# Check that the Pyomo model exists
pyM = esM.pyM

print(type(pyM))

<class 'pyomo.core.base.PyomoModel.ConcreteModel'>


In [13]:
# Print all Pyomo variables
print("PYOMO VARIABLES")
print("=" * 80)

for variable in pyM.component_objects(pyo.Var, active=True):
    print(f"\nVariable group: {variable.name}")
    print("-" * 80)

    for index in variable:
        variable_entry = variable[index]

        print(
            f"{variable.name}[{index}]"
            f" = {pyo.value(variable_entry)}"
            f", lower bound = {variable_entry.lb}"
            f", upper bound = {variable_entry.ub}"
        )

PYOMO VARIABLES

Variable group: cap_srcSnk
--------------------------------------------------------------------------------
cap_srcSnk[('Region_A', 'Source_A', 0)] = 8.666666666666668, lower bound = 0, upper bound = 20.0
cap_srcSnk[('Region_B', 'Source_B', 0)] = 0.0, lower bound = 0, upper bound = 20.0

Variable group: nbReal_srcSnk
--------------------------------------------------------------------------------
nbReal_srcSnk[('Region_A', 'Source_A', 0)] = 8.666666666666668, lower bound = 0, upper bound = None
nbReal_srcSnk[('Region_B', 'Source_B', 0)] = 0.0, lower bound = 0, upper bound = None

Variable group: nbInt_srcSnk
--------------------------------------------------------------------------------

Variable group: commisBin_srcSnk
--------------------------------------------------------------------------------

Variable group: op_srcSnk
--------------------------------------------------------------------------------
op_srcSnk[('Region_A', 'Source_A', 0, 0, 0)] = 6.44444444444444

### Reading the variables

- Some variable groups are empty (for example `nbInt_srcSnk`). FINE creates
  the group for every option, but fills it only if a component uses that
  option, here integer unit numbers.
- The demand is fixed through the variable bounds (lower bound = upper bound
  = demand), not through a constraint.
- `cap_srcSnk` of Source A is 8.67 MW: the largest load on Source A,
  2 MW own demand plus 6.67 MW sent to Region B in the last time step.

In [14]:
# Collect all active Pyomo constraints
constraint_records = []

for constraint_component in pyM.component_objects(
    pyo.Constraint,
    active=True,
):
    for index in constraint_component:
        constraint = constraint_component[index]

        if not constraint.active:
            continue

        constraint_records.append(
            {
                "constraint_group": constraint_component.name,
                "index": str(index),
                "expression": str(constraint.expr),
                "lower_bound": (
                    None if constraint.lower is None else str(constraint.lower)
                ),
                "body": str(constraint.body),
                "upper_bound": (
                    None if constraint.upper is None else str(constraint.upper)
                ),
            }
        )

constraints_df = pd.DataFrame(constraint_records)

print(
    f"Number of active constraint groups: "
    f"{constraints_df['constraint_group'].nunique()}"
)
print(f"Total number of active constraints: {len(constraints_df)}")

Number of active constraint groups: 10
Total number of active constraints: 32


In [15]:
# List the generated constraint groups and their sizes
constraint_group_summary = (
    constraints_df.groupby("constraint_group")
    .size()
    .rename("number_of_constraints")
    .reset_index()
)

constraint_group_summary

,constraint_group,number_of_constraints
0,ConstrCapToNbReal_srcSnk,2
1,ConstrCapToNbReal_trans,2
2,ConstrOperation3_srcSnk,6
3,ConstrOperation_trans,6
4,ConstrSymmetricalCapacity_trans,2
5,DecommConstrCapacityDevelopment_srcSnk,2
6,DecommConstrCapacityDevelopment_trans,2
7,InitialYear_srcSnk,2
8,InitialYear_trans,2
9,commodityBalanceConstraint,6


### Reading the constraints

Only 10 of FINE's constraint groups contain constraints in this model; the
others belong to options no component uses.

- `ConstrCapToNbReal`, `DecommConstrCapacityDevelopment`, `InitialYear`:
  bookkeeping. With a single investment period, they only set capacity equal
  to the built capacity and decommissioning to zero.
- `ConstrOperation3_srcSnk`: operation ≤ capacity × availability × hours per
  time step. With availability 1 and 1 h this reads `op ≤ cap`.
- `ConstrSymmetricalCapacity_trans`: the line has the same capacity in both directions.
- `ConstrOperation_trans`: the flows A→B and B→A **together** are limited by
  the capacity, so the line is not used in both directions at the same time.
- `commodityBalanceConstraint`: in every region and time step, supply equals
  demand. Imports enter with the factor 0.9 because of the line losses.

In [16]:
# Print every active constraint in a readable form
for number, row in constraints_df.iterrows():
    print(f"Constraint {number + 1}")
    print(f"Group: {row['constraint_group']}")
    print(f"Index: {row['index']}")
    print(f"Expression: {row['expression']}")
    print()

Constraint 1
Group: ConstrCapToNbReal_srcSnk
Index: ('Region_A', 'Source_A', 0)
Expression: cap_srcSnk[Region_A,Source_A,0]  ==  nbReal_srcSnk[Region_A,Source_A,0]

Constraint 2
Group: ConstrCapToNbReal_srcSnk
Index: ('Region_B', 'Source_B', 0)
Expression: cap_srcSnk[Region_B,Source_B,0]  ==  nbReal_srcSnk[Region_B,Source_B,0]

Constraint 3
Group: DecommConstrCapacityDevelopment_srcSnk
Index: ('Region_A', 'Source_A', 0)
Expression: decommis_srcSnk[Region_A,Source_A,0]  ==  0

Constraint 4
Group: DecommConstrCapacityDevelopment_srcSnk
Index: ('Region_B', 'Source_B', 0)
Expression: decommis_srcSnk[Region_B,Source_B,0]  ==  0

Constraint 5
Group: InitialYear_srcSnk
Index: ('Region_A', 'Source_A')
Expression: cap_srcSnk[Region_A,Source_A,0]  ==  commis_srcSnk[Region_A,Source_A,0] - decommis_srcSnk[Region_A,Source_A,0]

Constraint 6
Group: InitialYear_srcSnk
Index: ('Region_B', 'Source_B')
Expression: cap_srcSnk[Region_B,Source_B,0]  ==  commis_srcSnk[Region_B,Source_B,0] - decommis_srcSnk[

## Objective function

The objective is the total annual cost. Two things make it look unusual:

- Operating costs are divided by 0.000342 = 3 h / 8760 h. FINE scales the
  three modelled hours to a full year, which is why the costs are large.
- The factors 0.926 and 1.08 come from discounting with the 8 % interest
  rate; here they cancel out (0.926 × 1.08 = 1).

Investment costs enter as annuities: the investment spread over the
economic lifetime, for example 5 €/MW × 0.102 = 0.509 €/MW per year for the
sources (20 years, 8 %).

In [17]:
# Print the objective function
print("OBJECTIVE FUNCTION")
print("=" * 80)

for objective in pyM.component_objects(pyo.Objective, active=True):
    print(f"\nName: {objective.name}")
    print(f"Sense: {objective.sense}")
    print(f"Expression:\n{objective.expr}")
    print(f"\nOptimal value: {pyo.value(objective)}")

OBJECTIVE FUNCTION

Name: Obj
Sense: minimize
Expression:
(0.0*op_srcSnk[Region_A,Electricity_demand,0,0,0] + 0.0*op_srcSnk[Region_A,Electricity_demand,0,0,1] + 0.0*op_srcSnk[Region_A,Electricity_demand,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (op_srcSnk[Region_A,Source_A,0,0,0] + op_srcSnk[Region_A,Source_A,0,0,1] + op_srcSnk[Region_A,Source_A,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (10.0*op_srcSnk[Region_B,Source_B,0,0,0] + 10.0*op_srcSnk[Region_B,Source_B,0,0,1] + 10.0*op_srcSnk[Region_B,Source_B,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (0.0*op_srcSnk[Region_B,Electricity_demand,0,0,0] + 0.0*op_srcSnk[Region_B,Electricity_demand,0,0,1] + 0.0*op_srcSnk[Region_B,Electricity_demand,0,0,2])/0.00034246575342465754*0.9259259259259267*1.08 + (0.5092610441157533*commis_srcSnk[Region_A,Source_A,0] + 0.5092610441157533*commis_srcSnk[Region_B,Source_B,0] + 0.0*commis_srcSnk[Region_A,Source_A,0] + 0.0*commis_srcSnk[Region_B,Source_B,0]) + (0.0*op

## Results

As expected, only Source A is built. The line needs 6.67 MW, not 6 MW: to
deliver the 6 MW peak demand of Region B, 6 / 0.9 = 6.67 MW must be sent.
FINE reports half of the line cost for each direction.

In [18]:
# Display optimization summaries
source_summary = esM.getOptimizationSummary(
    "SourceSinkModel",
    outputLevel=2,
)

source_summary

Region_A Region_B
Component          Property         Unit                           
Electricity_demand operation        [MW*h]             7.0     15.0
                   operation_annual [MW*h/a]       20440.0  43800.0
Source_A           NPVcontribution  [Euro]    69111.080262      0.0
                   TAC              [Euro/a]  69111.080262      0.0
                   capacity         [MW]          8.666667      NaN
                   capexCap         [Euro/a]      4.413596      NaN
                   commissioning    [MW]          8.666667      NaN
                   invest           [Euro]       43.333333      NaN
                   operation        [MW*h]       23.666667      NaN
                   operation_annual [MW*h/a]  69106.666667      NaN
                   opexOp           [Euro/a]  69106.666667      NaN

In [19]:
# Display optimization summaries
transmission_summary = esM.getOptimizationSummary(
    "TransmissionModel",
    outputLevel=2,
)

transmission_summary

Region_A      Region_B
Component        Property         Unit     locationIn                         
Electricity_grid NPVcontribution  [Euro]   Region_A            0      2.795339
                                           Region_B     2.795339             0
                 TAC              [Euro/a] Region_A            0      2.795339
                                           Region_B     2.795339             0
                 capacity         [MW]     Region_A          NaN      6.666667
                                           Region_B     6.666667           NaN
                 capexCap         [Euro/a] Region_A          NaN      2.795339
                                           Region_B     2.795339           NaN
                 commissioning    [MW]     Region_A          NaN      6.666667
                                           Region_B     6.666667           NaN
                 invest           [Euro]   Region_A          NaN     33.333333
                                           Region_B    33.333333           NaN
                 operation        [MW*h]   Region_A          NaN     16.666667
                 operation_annual [MW*h/a] Region_A          NaN  48666.666667

## Show the constraints as equations

Pyomo's LaTeX printer can display each constraint as an equation. FINE's
constraint rules cannot be printed in general form (with "for all" indices),
so each constraint is printed for each index separately.

In [20]:
try:
    from pyomo.contrib.latex_printer import latex_printer

    latex_printer_available = True
    print("Pyomo LaTeX printer is available.")
except ImportError:
    latex_printer_available = False
    print("Pyomo LaTeX printer is not available in this Pyomo installation.")

Pyomo LaTeX printer is available.


In [21]:
# Render all active constraints as LaTeX, grouped by constraint group
from IPython.display import Markdown, Math, display

if latex_printer_available:
    for constraint_component in pyM.component_objects(
        pyo.Constraint,
        active=True,
    ):
        active_constraints = [
            constraint_component[index]
            for index in constraint_component
            if constraint_component[index].active
        ]

        # Skip constraint groups without any constraints in this model
        if not active_constraints:
            continue

        display(Markdown(f"**{constraint_component.name}**"))

        for constraint in active_constraints:
            # MathJax expects the bare expression without the equation environment
            latex_expression = (
                latex_printer(constraint.expr)
                .replace(r"\begin{equation}", "")
                .replace(r"\end{equation}", "")
                .strip()
            )
            display(Math(latex_expression))

**ConstrCapToNbReal_srcSnk**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**DecommConstrCapacityDevelopment_srcSnk**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

... (32 outputs left out)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Summary

FINE creates the optimization problem automatically from the added
components. For this system it contains 32 constraints:

- bookkeeping constraints that link capacity, built capacity and number of units,
- limits on the operation of sources and of the power line,
- a commodity balance for every region and time step.

The demand is not a constraint: FINE fixes it through the bounds of the
demand variables.